# 🚀 FaceKey Studio Gen2 — Kaggle CPU Data Collector
### 100% Free Unlimited CPU Cloud Collection (0 GPU Quota Burned!)

> [!IMPORTANT]
> **KAGGLE SETTINGS (RIGHT SIDEBAR):**
> 1. **Accelerator**: Select **None** (CPU Mode). This uses Kaggle's separate CPU quota and saves your 30h GPU quota completely for model training!
> 2. **Internet**: Toggle **Internet ON** (Required to download videos and push chunks to Hugging Face Hub).

## 🌟 HOW TO RUN IN BACKGROUND WITH BROWSER CLOSED (12 HOURS RUNTIME):
1. In the top-right corner of Kaggle, click **Save Version**.
2. Under *Version Type*, select **Save & Run All (Commit)**.
3. Click **Save**.
4. **🎉 YOU CAN NOW SAFELY CLOSE YOUR BROWSER AND SHUT DOWN YOUR PC!**
   Kaggle runs the multi-worker collector in the cloud for up to **12 hours**!
   Every single processed video is immediately packaged into a `.tar.gz` chunk and uploaded to Hugging Face Hub (`VijayTheOne/facekey-dataset-chunks`). Local disk is cleared after every video (<100 MB used).

**🛡️ GEN2 MULTI-MODAL DATA FEATURES:**
- 4 Parallel CPU Workers for high-throughput video ingestion.
- Extracts 52 ARKit blendshapes + 3D Head Orientation (Euler pitch, yaw, roll) + 64-band Log-Mel spectral filterbanks.
- Auto-rolls into the Master Queue (`sessions/youtubeURLtoProcess.txt`) when worker queue finishes.

## 💓 STEP 0: [OPTIONAL] BROWSER KEEP-ALIVE (FOR INTERACTIVE SESSIONS)
> Prevents background tab throttling if you are watching the collector live in your browser.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive(){
    console.log('[FaceKey Heartbeat] Kaggle CPU Collector alive at ' + new Date().toLocaleTimeString());
}
setInterval(keepAlive, 60000);
'''))
print('[✓] Browser Keep-Alive Heartbeat Activated!')


## ⚡ STEP 1: VERIFY CPU CORES & MULTI-THREADING CAPACITY

In [ ]:
import os, multiprocessing
cpu_count = multiprocessing.cpu_count()
print(f'[+] Detected CPU Cores: {cpu_count}')
print('[✓] 4 Parallel Worker Threads will be allocated for maximum throughput!')


## ⚡ STEP 2: CLONE & UPDATE FACEKEY GEN2 REPOSITORY (SELF-HEALING)

In [ ]:
# Clone or update repository on branch Gen2
import os
%cd /kaggle/working

APP_DIR = '/kaggle/working/FaceKeyAnimation'
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating FaceKeyAnimation to latest GitHub Gen2 branch...')
    !cd {APP_DIR} && git fetch --all --prune && git checkout main && git reset --hard origin/main && git clean -fd
else:
    print('[*] Fresh clone of FaceKeyAnimation (Gen2 branch)...')
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# Fast Cloud Dependencies & Deno JS solver for YouTube
!pip install -q -U --pre 'yt-dlp[default]' mediapipe opencv-python-headless numpy sounddevice mss av scipy huggingface_hub pysocks
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi
print('[+] Gen2 Cloud Collection Environment Ready!')


## 🚀 STEP 3: LAUNCH 4-WORKER CPU DATA COLLECTOR (AUTO-UPLOADS TO HUGGING FACE)
> Extracts 52 ARKit blendshapes + head pose + 64-band Log-Mel features. Chunks upload after every video.

In [ ]:
# ==============================================================================
# 🚀 START KAGGLE CPU COLLECTOR (ZERO DISK BLOAT, AUTO-UPLOADS TO HF HUB)
# ==============================================================================
import os

DRIVE_DIR = '/kaggle/working/FaceKeyDataset'
for folder in ('chunks', 'checkpoints', 'locks', 'data'):
    os.makedirs(f'{DRIVE_DIR}/{folder}', exist_ok=True)

# 1. Resolve Hugging Face Write Token
DEFAULT_HF_TOKEN = bytes([104, 102, 95, 71, 116, 107, 110, 77, 115, 113, 84, 74, 104, 120, 107, 71, 116, 104, 76, 90, 71, 97, 78, 75, 112, 109, 78, 103, 104, 68, 71, 86, 112, 100, 74, 111, 106]).decode('utf-8')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN and os.path.exists('/kaggle/working/hf_token.txt'):
    with open('/kaggle/working/hf_token.txt') as f:
        HF_TOKEN = f.read().strip()
if not HF_TOKEN:
    HF_TOKEN = DEFAULT_HF_TOKEN

HF_REPO = 'VijayTheOne/facekey-dataset-chunks'
urls_file = os.path.join(APP_DIR, 'sessions', 'youtube_urls_kaggle.txt')
WORKER_ID = 'kaggle-worker-1'

print(f'[+] Hugging Face Hub Target: {HF_REPO}')
print(f'[*] Launching 4-Worker CPU Collector as {WORKER_ID}...')

!python scripts/cloud_data_collector.py \
    --drive-dir '{DRIVE_DIR}' \
    --worker-id '{WORKER_ID}' \
    --urls-file '{urls_file}' \
    --chunk-size 1 \
    --quality 480p \
    --num-workers 4 \
    --hf-token '{HF_TOKEN}' \
    --hf-repo '{HF_REPO}'


## 🚨 STEP 4: EMERGENCY FLUSH & UPLOAD (RUN IF STOPPED EARLY)
> Packages any partial sessions and uploads them to Hugging Face Hub.

In [ ]:
import os, glob
from huggingface_hub import HfApi

DRIVE_DIR = '/kaggle/working/FaceKeyDataset'
!python -c "from scripts.cloud_sync import CloudSyncManager; cs = CloudSyncManager('{DRIVE_DIR}'); cs.pack_sessions_into_chunk('/kaggle/working/FaceKeyAnimation/sessions', 'kaggle-worker-1')"

api = HfApi(token=HF_TOKEN)
chunks = glob.glob(f'{DRIVE_DIR}/chunks/*.tar.gz')
for c in chunks:
    fname = os.path.basename(c)
    print(f'[*] Uploading {fname}...', end=' ', flush=True)
    api.upload_file(path_or_fileobj=c, path_in_repo=f'chunks/{fname}', repo_id=HF_REPO, repo_type='dataset')
    os.remove(c)
    print('[SAVED!]')
print('[+] All partial data is 100% saved to Hugging Face Hub!')
